In [ ]:
# Imports
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Literal
import outlines

import torch
import torch.nn.functional as F
from pydantic import BaseModel, Field, create_model
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA_DIR = Path("data/ToolVerifier")
EMBED_ARCHITECTURE = "normal"
EMBED_LOSS = "circle"
CHECKPOINT_PATH = DATA_DIR / "output" / EMBED_ARCHITECTURE / EMBED_LOSS / "best.pt"
TOOLS_PATH = DATA_DIR / "tools.json"

from training import embed_texts, load_checkpoint_bundle

QWEN_MODEL_NAME = "Qwen/Qwen3.5-27B"
QWEN_DEVICE = torch.device("cuda:1")
EMBED_DEVICE = torch.device("cuda:3")
QWEN_DTYPE = torch.bfloat16

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires CUDA.")

if torch.cuda.device_count() < 2:
    raise RuntimeError("This notebook requires at least 2 CUDA GPUs.")

with TOOLS_PATH.open("r", encoding="utf-8") as handle:
    TOOL_REGISTRY = json.load(handle)["tools"]

TOOL_BY_NAME = {tool["name"]: tool for tool in TOOL_REGISTRY}


/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Constants
embedding_bundle = load_checkpoint_bundle(CHECKPOINT_PATH, device=str(EMBED_DEVICE))
EMBED_MODEL = embedding_bundle["model"]
EMBED_TOKENIZER = embedding_bundle["tokenizer"]
EMBED_MAX_LENGTH = int(embedding_bundle["max_length"])
TOOL_NAMES = list(embedding_bundle["tool_names"])
TOOL_CENTROIDS = F.normalize(embedding_bundle["centroids"], dim=-1)

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token or qwen_tokenizer.unk_token

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    trust_remote_code=True,
    dtype=QWEN_DTYPE,
    device_map={"": str(QWEN_DEVICE)},
)
qwen_model.eval()
qwen_model.generation_config.pad_token_id = qwen_tokenizer.pad_token_id
structured_qwen = outlines.from_transformers(qwen_model, qwen_tokenizer)

TOP_K = 5
MAX_ACTIONS = 10
PLANNER_MAX_NEW_TOKENS = 96
DISPATCHER_MAX_NEW_TOKENS = 160

USER_REQUEST = "Get me the top news for the day and email it to akrik@umich.edu"

print(f"Embedding checkpoint: {CHECKPOINT_PATH}")
print(f"Loaded tools: {len(TOOL_NAMES)}")
print(f"Embedding device: {EMBED_DEVICE}")
print(f"Qwen model: {QWEN_MODEL_NAME}")
print(f"Qwen device: {QWEN_DEVICE}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1614.54it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 112530.11it/s]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 851/851 [00:09<00:00, 85.50it/s, Materializing param=model.norm.weight]                               


Embedding checkpoint: data/ToolVerifier/output/normal/circle/best.pt
Loaded tools: 124
Embedding device: cuda:3
Qwen model: Qwen/Qwen3.5-27B
Qwen device: cuda:1


In [3]:
# Harness
class PlanOutput(BaseModel):
    actions: list[str] = Field(..., min_length=1, max_length=MAX_ACTIONS)


def normalize_outline_text(value: str) -> str:
    return re.sub(r"\s+", " ", str(value)).strip()


def normalize_plan_actions(actions: list[str]) -> list[str]:
    deduped: list[str] = []
    for action in actions:
        cleaned = normalize_outline_text(action)
        if cleaned and cleaned not in deduped:
            deduped.append(cleaned)
    return deduped[:MAX_ACTIONS] or [normalize_outline_text(USER_REQUEST)]


def build_plan_block(actions: list[str]) -> str:
    lines = ["<plan>"]
    for action in actions:
        lines.append(f"  <action><len:{len(action)}>{action}</len></action>")
    lines.append("</plan>")
    return "\n".join(lines)


def retrieve_tools(actions: list[str], top_k: int = TOP_K) -> list[dict]:
    action_embeddings = embed_texts(
        model=EMBED_MODEL,
        tokenizer=EMBED_TOKENIZER,
        texts=actions,
        device=EMBED_DEVICE,
        max_length=EMBED_MAX_LENGTH,
        batch_size=min(len(actions), 8),
        progress_desc="Embedding plan actions",
    )
    action_embeddings = F.normalize(action_embeddings.to(TOOL_CENTROIDS.device), dim=-1)
    score_matrix = action_embeddings @ TOOL_CENTROIDS.T

    rows = []
    k = min(top_k, len(TOOL_NAMES))
    for action, scores in zip(actions, score_matrix):
        top_scores, top_indices = torch.topk(scores, k=k)
        candidates = []
        for score, index in zip(top_scores.tolist(), top_indices.tolist()):
            candidates.append({
                "tool": TOOL_NAMES[index],
                "score": float(score),
            })
        rows.append({"action": action, "candidates": candidates})
    return rows


def schema_type_to_annotation(spec: dict[str, Any]) -> Any:
    schema_type = spec.get("type", "string")
    enum_values = spec.get("enum")
    if enum_values:
        return Literal.__getitem__(tuple(enum_values))
    if schema_type == "string":
        return str
    if schema_type == "integer":
        return int
    if schema_type == "number":
        return float
    if schema_type == "boolean":
        return bool
    if schema_type == "array":
        item_spec = spec.get("items", {})
        item_annotation = schema_type_to_annotation(
            item_spec if isinstance(item_spec, dict) else {"type": "string"}
        )
        return list[item_annotation]
    return Any


def build_dispatch_output_model(tool_name: str, schema: dict[str, Any]) -> type[BaseModel]:
    properties = schema.get("properties", {})
    required = set(schema.get("required", []))
    fields = {}

    for name, raw_spec in properties.items():
        spec = raw_spec if isinstance(raw_spec, dict) else {"type": "string"}
        annotation = schema_type_to_annotation(spec)
        if name in required:
            fields[name] = (annotation, Field(...))
            continue
        fields[name] = (annotation | None, Field(default=spec.get("default", None)))

    safe_name = "".join(part.capitalize() for part in re.split(r"[^0-9A-Za-z]+", tool_name) if part) or "Tool"
    return create_model(f"{safe_name}Dispatch", **fields)


def build_generation_kwargs(max_new_tokens: int) -> dict[str, Any]:
    return {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": qwen_tokenizer.pad_token_id,
        "eos_token_id": qwen_tokenizer.eos_token_id,
        "do_sample": False,
    }


def normalize_dispatch_arguments(arguments: dict[str, Any]) -> dict[str, Any]:
    normalized = {}
    for name, value in arguments.items():
        if value is None:
            continue
        if isinstance(value, str):
            normalized[name] = normalize_outline_text(value)
        elif isinstance(value, list):
            normalized[name] = [
                normalize_outline_text(item) if isinstance(item, str) else item
                for item in value
            ]
        else:
            normalized[name] = value
    return normalized


def build_dispatch_block(tool_name: str, arguments: dict[str, Any]) -> str:
    lines = ["<dispatch>", f"  <tool><len:{len(tool_name)}>{tool_name}</len></tool>"]
    for name, value in arguments.items():
        rendered = value if isinstance(value, str) else json.dumps(value)
        lines.append(f'  <arg name="{name}"><len:{len(rendered)}>{rendered}</len></arg>')
    lines.append("</dispatch>")
    return "\n".join(lines)


planner_generator = outlines.Generator(structured_qwen, PlanOutput)
dispatch_generator_cache: dict[str, tuple[type[BaseModel], Any]] = {}


def get_dispatch_generator(tool_name: str, schema: dict[str, Any]):
    cache_key = f"{tool_name}:{json.dumps(schema, sort_keys=True)}"
    cached = dispatch_generator_cache.get(cache_key)
    if cached is None:
        output_model = build_dispatch_output_model(tool_name, schema)
        cached = (output_model, outlines.Generator(structured_qwen, output_model))
        dispatch_generator_cache[cache_key] = cached
    return cached


In [4]:
# Planner
planner_prompt = f"""You are the planner inside NTILC.
Return a plan with an actions field containing at most {MAX_ACTIONS} short atomic natural language actions.
Requirements:
- Actions must be plain English task steps, not tool calls.
- Do not use tool names, API names, function names, schema fields, or argument names.
- Each action must be a single line.
- Include the final user-facing step when needed.

Example:
User request: get me the top news today and email to akrik@umich.edu
actions:
- get today's date
- get news with today's date
- compose email with todays news and send to akrik@umich.edu

Do not include commentary, reasoning, numbering, or extra fields.

User request:
{USER_REQUEST}
"""

print("Planner JSON:\n")
raw_plan_json = planner_generator(planner_prompt, **build_generation_kwargs(PLANNER_MAX_NEW_TOKENS))
print(raw_plan_json)

plan_payload = PlanOutput.model_validate_json(raw_plan_json)
plan_actions = normalize_plan_actions(plan_payload.actions)
plan_block = build_plan_block(plan_actions)

print("\nParsed actions:")
for index, action in enumerate(plan_actions, start=1):
    print(f"{index}. {action}")

print("\nPlan block:")
print(plan_block)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Planner JSON:

{ "actions": [ "get today's date", "get top news for today", "compose email with today's top news", "send email to akrik@umich.edu" ] }

Parsed actions:
1. get today's date
2. get top news for today
3. compose email with today's top news
4. send email to akrik@umich.edu

Plan block:
<plan>
  <action><len:16>get today's date</len></action>
  <action><len:22>get top news for today</len></action>
  <action><len:35>compose email with today's top news</len></action>
  <action><len:29>send email to akrik@umich.edu</len></action>
</plan>


In [5]:
# Embedding Look Up
retrieval_rows = retrieve_tools(plan_actions, top_k=TOP_K)

for row in retrieval_rows:
    print(f"\nAction: {row['action']}")
    for rank, candidate in enumerate(row["candidates"], start=1):
        print(f"  {rank}. {candidate['tool']:<24} score={candidate['score']:.4f}")



Action: get today's date
  1. get_today_date           score=0.8305
  2. get_date                 score=0.7824
  3. calendar_date            score=0.3393
  4. get_location             score=0.2796
  5. date_format_convertor    score=0.2659

Action: get top news for today
  1. get_top_5_news           score=0.6810
  2. news_top_stories         score=0.6689
  3. latest_news              score=0.5745
  4. get_news                 score=0.4122
  5. generate_bank_account_number score=0.2621

Action: compose email with today's top news
  1. news_top_stories         score=0.6011
  2. get_top_5_news           score=0.3948
  3. get_date                 score=0.3604
  4. generate_bank_account_number score=0.3254
  5. book_review_by_day       score=0.3145

Action: send email to akrik@umich.edu
  1. hotel_reservations       score=0.3560
  2. get_calendar_meeting     score=0.3449
  3. get_job                  score=0.2962
  4. get_bank_account_number  score=0.2842
  5. job_numbers              sco

In [6]:
# Tool Mapping from Embedding to Schema
mapped_steps = []

for row in retrieval_rows:
    selected_tool_name = row["candidates"][0]["tool"]
    selected_schema = TOOL_BY_NAME[selected_tool_name]["parameters"]
    mapped_steps.append({
        "action": row["action"],
        "tool_name": selected_tool_name,
        "schema": selected_schema,
    })

for step in mapped_steps:
    print(f"\nAction: {step['action']}")
    print(f"Selected tool: {step['tool_name']}")
    print("Schema:")
    print(json.dumps(step["schema"], indent=2))



Action: get today's date
Selected tool: get_today_date
Schema:
{
  "type": "object",
  "properties": {
    "format": {
      "type": "string",
      "default": "YYYY-MM-DD"
    }
  },
  "required": []
}

Action: get top news for today
Selected tool: get_top_5_news
Schema:
{
  "type": "object",
  "properties": {
    "location": {
      "type": "string",
      "default": "global"
    },
    "date": {
      "type": "string",
      "default": "today"
    }
  },
  "required": []
}

Action: compose email with today's top news
Selected tool: news_top_stories
Schema:
{
  "type": "object",
  "properties": {
    "date": {
      "type": "string",
      "default": "today"
    },
    "location": {
      "type": "string",
      "default": null
    }
  },
  "required": [
    "location"
  ]
}

Action: send email to akrik@umich.edu
Selected tool: hotel_reservations
Schema:
{
  "type": "object",
  "properties": {
    "hotel_name": {
      "type": "string",
      "default": "Grand Hotel"
    },
    "loc

In [7]:
# Dispatcher
dispatch_blocks = []

for step_index, step in enumerate(mapped_steps, start=1):
    dispatcher_prompt = f"""You are the NTILC dispatcher.
Return only the arguments for the selected tool.
Use the provided schema exactly.
Do not include explanation or extra fields.

User request:
{USER_REQUEST}

Current action:
{step['action']}

Selected tool:
{step['tool_name']}

Tool schema:
{json.dumps(step['schema'], indent=2)}
"""

    output_model, dispatch_generator = get_dispatch_generator(step["tool_name"], step["schema"])

    print(f"\n===== Step {step_index}: {step['tool_name']} =====")
    print("Dispatcher JSON:\n")
    raw_dispatch_json = dispatch_generator(
        dispatcher_prompt,
        **build_generation_kwargs(DISPATCHER_MAX_NEW_TOKENS),
    )
    print(raw_dispatch_json)

    dispatch_payload = output_model.model_validate_json(raw_dispatch_json)
    dispatch_arguments = normalize_dispatch_arguments(
        dispatch_payload.model_dump(exclude_none=True)
    )
    dispatch_block = build_dispatch_block(step["tool_name"], dispatch_arguments)
    dispatch_blocks.append(dispatch_block)

    print("\nDispatch block:")
    print(dispatch_block)



===== Step 1: get_today_date =====
Dispatcher JSON:

{ "format": "YYYY-MM-DD" }

Dispatch block:
<dispatch>
  <tool><len:14>get_today_date</len></tool>
  <arg name="format"><len:10>YYYY-MM-DD</len></arg>
</dispatch>

===== Step 2: get_top_5_news =====
Dispatcher JSON:

{ "location": "global", "date": "today" }

Dispatch block:
<dispatch>
  <tool><len:14>get_top_5_news</len></tool>
  <arg name="location"><len:6>global</len></arg>
  <arg name="date"><len:5>today</len></arg>
</dispatch>

===== Step 3: news_top_stories =====
Dispatcher JSON:

{ "location": "US" }

Dispatch block:
<dispatch>
  <tool><len:16>news_top_stories</len></tool>
  <arg name="date"><len:5>today</len></arg>
  <arg name="location"><len:2>US</len></arg>
</dispatch>

===== Step 4: hotel_reservations =====
Dispatcher JSON:

{ "hotel_name": "Grand Hotel", "location": "New York", "check_in_date": "2023-10-01", "check_out_date": "2023-10-05", "room_type": "Standard", "guest_name": "John Doe" }

Dispatch block:
<dispatch>
  

In [8]:
# Summary
print("User query:\n")
print(USER_REQUEST)

print("\nPlan:\n")
print(plan_block)

print("\nDispatch blocks by action:")
for step_index, (step, dispatch_block) in enumerate(zip(mapped_steps, dispatch_blocks, strict=True), start=1):
    print(f"\nAction {step_index}: {step['action']}")
    print(dispatch_block)


User query:

Get me the top news for the day and email it to akrik@umich.edu

Plan:

<plan>
  <action><len:16>get today's date</len></action>
  <action><len:22>get top news for today</len></action>
  <action><len:35>compose email with today's top news</len></action>
  <action><len:29>send email to akrik@umich.edu</len></action>
</plan>

Dispatch blocks by action:

Action 1: get today's date
<dispatch>
  <tool><len:14>get_today_date</len></tool>
  <arg name="format"><len:10>YYYY-MM-DD</len></arg>
</dispatch>

Action 2: get top news for today
<dispatch>
  <tool><len:14>get_top_5_news</len></tool>
  <arg name="location"><len:6>global</len></arg>
  <arg name="date"><len:5>today</len></arg>
</dispatch>

Action 3: compose email with today's top news
<dispatch>
  <tool><len:16>news_top_stories</len></tool>
  <arg name="date"><len:5>today</len></arg>
  <arg name="location"><len:2>US</len></arg>
</dispatch>

Action 4: send email to akrik@umich.edu
<dispatch>
  <tool><len:18>hotel_reservations</